# 04. Module Bus & Events — deep dive

> **Read [`03-policy-and-modules.ipynb`](./03-policy-and-modules.ipynb) first.** That notebook covered the *shape* of the bus — the `Module` Protocol, `ModuleBus`, `ModuleContext`, the `EventContext`, the policy/bus interplay, and `ToolVetoedError`. **This** notebook is the deep-dive: the *vocabulary* of events that ArcAgent actually emits during a run, the lifecycle that ties modules to the bus, the difference between `HookEntry`, `BackgroundTaskEntry`, and `LifecycleEntry`, three operator-facing module patterns, and the failure modes you must understand before shipping a module to a federal deployment.

You will learn:

- The complete event taxonomy `ArcAgent` emits — names, when each fires, and the data payload.
- The full module lifecycle — registration → startup ordering → handler dispatch → shutdown teardown.
- When to reach for a `HookEntry` vs a `BackgroundTaskEntry` vs a `LifecycleEntry` — three concepts from `arcagent.core.capability_registry` that look similar but solve different problems.
- Three production patterns: a **metrics** module, a **structured logging** module that fans out to a sink, and a **streaming** module that forwards events to an external transport.
- Failure modes — what happens when a module raises during startup, raises in an event handler, or times out — and how the bus contains the blast radius.
- How to compose multiple modules and reason about isolation, ordering, and debugging when something goes wrong.

All cells run with **no API key, no real LLM, no real network**. The bus is in-process, events are synthetic, and any "external" sink is an in-memory list.

## 1. Setup

Standard Arc walkthrough boilerplate — same first cell every notebook in the series uses. After it runs, every package's `src/` (and `tests/`, where present) is on `sys.path`, so `import arcagent`, `import arctrust`, and `import arcrun` resolve from the in-repo source.

> **Reminder.** This is a deep-dive — read [`03-policy-and-modules.ipynb`](./03-policy-and-modules.ipynb) first to understand the bus *surface*. This notebook builds on that foundation.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")


Pull in the bus primitives, the error types, and a couple of `unittest.mock` helpers we'll use to fake the heavyweight dependencies of `ModuleContext` (`AgentTelemetry`, full `ArcAgentConfig`, etc.) without spinning up a full `ArcAgent`.

In [ ]:
import asyncio
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, fields
from typing import Any
from unittest.mock import MagicMock

import arcagent
from arcagent import ModuleBusError, ToolVetoedError
from arcagent.core.module_bus import (
    EventContext,
    Module,
    ModuleBus,
    ModuleContext,
)

print("arcagent          :", arcagent.__version__)
print("ModuleBus         :", ModuleBus.__name__)
print("Module is Protocol:", Module.__module__)


For a couple of the patterns below we'll also reach for the capability-registry types — `HookEntry`, `BackgroundTaskEntry`, `LifecycleEntry`. These don't sit on the bus directly; they're how the **capability loader** describes the *shape* of an extension before bridging it onto the bus. Section 4 is dedicated to the distinction.

In [ ]:
from arcagent.core.capability_registry import (
    BackgroundTaskEntry,
    HookEntry,
    LifecycleEntry,
)

print("HookEntry            :", HookEntry.__name__)
print("BackgroundTaskEntry  :", BackgroundTaskEntry.__name__)
print("LifecycleEntry       :", LifecycleEntry.__name__)


## 2. The event taxonomy

ArcAgent emits a fixed set of bus events from its core. Bridges (arcrun, arcllm) translate their own event types into bus events as well — the canonical surface is below.

| Event | When it fires | Payload (keys) | Veto allowed? |
|---|---|---|---|
| `agent:init` | At end of `ArcAgent.startup()` after `agent:ready`. | `config` (agent name) | no |
| `agent:ready` | Right before `agent:init`, with deferred-binding callbacks for hooks that want to invoke the agent. | `run_fn`, `chat_fn`, `run_async_fn`, `chat_async_fn` | no |
| `agent:pre_respond` | Just before the run loop is launched (one per `run` / `chat` / `chat_stream`). | `task` | no |
| `agent:post_respond` | After the run loop returns, before the agent finalises the session. | `result`, `messages`, `session_id` | no |
| `agent:error` | When the run loop raises. | `task`, `error`, `error_type` | no |
| `agent:pre_tool` | Inside `ToolRegistry._create_wrapped_execute`, after policy ALLOW, before tool execution. **This is the veto seam.** | `tool`, `args` | yes — `ctx.veto(reason)` raises `ToolVetoedError` |
| `agent:post_tool` | After successful tool execution. | `tool`, `result`, `duration` | no |
| `agent:pre_plan` | Bridged from arcrun `turn.start`. | arcrun event data | no |
| `agent:post_plan` | Bridged from arcrun `turn.end`. | arcrun event data | no |
| `agent:assemble_prompt` | When `ContextManager.assemble_system_prompt` runs — modules can inject sections. | `sections`, `workspace` | no (modules mutate `sections` in-place via the snapshot) |
| `agent:pre_compaction` | When token usage crosses `compact_threshold`. | `messages`, `ratio` | no |
| `agent:settings_changed` | When `SettingsManager.set` mutates a runtime-mutable key. | `key`, `old_value`, `new_value` | no |
| `agent:tools_reloaded` | After `ArcAgent.reload()` re-registers the capability surface. | `{}` | no |
| `agent:shutdown` | First step of `ArcAgent.shutdown()`, before per-component teardown. | `{}` | no |
| `capability:added` | After the capability registry registers a new tool/skill/hook/task/lifecycle. | `kind`, `name`, `version`, `source_path`, `scan_root`, … | no |
| `capability:replaced` | When a registration replaces an existing entry of the same name. | `kind`, `name`, `version`, `previous_version`, … | no |
| `capability:removed` | When `unregister(...)` succeeds. | `kind`, `name`, `version` | no |
| `capability:registration_failed` | The capability loader rejected an entry (validation/conflict). | `path`, `reason`, … | no |
| `capability:registration_warning` | Soft warning from loader (deprecation, fallback). | `path`, `reason`, … | no |
| `capability:setup_failed` | A `LifecycleEntry.instance.setup(ctx)` raised. | `name`, `error`, … | no |
| `llm:call_complete` | Bridged from arcllm `llm_call` traces (and arcrun's mirror). | trace fields incl. `model`, `provider`, `agent_label` | no |
| `llm:config_change` | Bridged from arcllm `config_change`. | trace fields | no |
| `llm:circuit_change` | Bridged from arcllm `circuit_change`. | trace fields | no |

The **only veto seam** is `agent:pre_tool`. Every other emission is observe-only — handlers can read, log, mutate the snapshot dict (it doesn't leak back), forward to a sink, or schedule background work, but they cannot stop the operation.

Let's introspect this in code by registering a *logger module* that captures every event ArcAgent could conceivably emit, then drive synthetic emissions through the same `ModuleBus` and look at the catch list. We won't run a real `ArcAgent` (that requires keys, a workspace, and a full config tree); we drive the bus directly so each event is reproducible and deterministic.

In [ ]:
# Synthetic taxonomy probe — register one handler per known event,
# emit a sample of each, see what flows through.

KNOWN_EVENTS: tuple[str, ...] = (
    "agent:init",
    "agent:ready",
    "agent:pre_respond",
    "agent:post_respond",
    "agent:error",
    "agent:pre_tool",
    "agent:post_tool",
    "agent:pre_plan",
    "agent:post_plan",
    "agent:assemble_prompt",
    "agent:pre_compaction",
    "agent:settings_changed",
    "agent:tools_reloaded",
    "agent:shutdown",
    "capability:added",
    "capability:replaced",
    "capability:removed",
    "capability:registration_failed",
    "capability:registration_warning",
    "capability:setup_failed",
    "llm:call_complete",
    "llm:config_change",
    "llm:circuit_change",
)

probe_log: list[tuple[str, dict[str, Any]]] = []


async def probe_handler(ctx: EventContext) -> None:
    probe_log.append((ctx.event, dict(ctx.data)))


probe_bus = ModuleBus()
for ev in KNOWN_EVENTS:
    probe_bus.subscribe(ev, probe_handler, module_name="probe")

print("probe handlers registered:", sum(probe_bus.handler_count(e) for e in KNOWN_EVENTS))


Now drive synthetic emissions through the bus — same names, same payload shapes the real components would use. The probe captures the catalog.

In [ ]:
SAMPLE_PAYLOADS: dict[str, dict[str, Any]] = {
    "agent:init": {"config": "demo-agent"},
    "agent:ready": {"run_fn": "<callable>", "chat_fn": "<callable>"},
    "agent:pre_respond": {"task": "summarise the meeting notes"},
    "agent:post_respond": {"result": "<LoopResult>", "messages": [], "session_id": "s-1"},
    "agent:error": {"task": "...", "error": "boom", "error_type": "RuntimeError"},
    "agent:pre_tool": {"tool": "read_file", "args": {"path": "/etc/hosts"}},
    "agent:post_tool": {"tool": "read_file", "result": "127.0.0.1 ...", "duration": 0.012},
    "agent:pre_plan": {"turn": 1},
    "agent:post_plan": {"turn": 1, "actions": 2},
    "agent:assemble_prompt": {"sections": {"identity": "..."}, "workspace": str(Path.cwd() / "_demo_workspace")},
    "agent:pre_compaction": {"messages": [], "ratio": 0.82},
    "agent:settings_changed": {"key": "context.max_tokens", "old_value": 200_000, "new_value": 400_000},
    "agent:tools_reloaded": {},
    "agent:shutdown": {},
    "capability:added": {"kind": "tool", "name": "search", "version": "1.0.0"},
    "capability:replaced": {"kind": "skill", "name": "writer", "version": "1.1.0", "previous_version": "1.0.0"},
    "capability:removed": {"kind": "tool", "name": "old_search", "version": "0.9.0"},
    "capability:registration_failed": {"path": "skills/bad/SKILL.md", "reason": "schema invalid"},
    "capability:registration_warning": {"path": "skills/old/SKILL.md", "reason": "deprecated field"},
    "capability:setup_failed": {"name": "memory", "error": "vault unreachable"},
    "llm:call_complete": {"event_type": "llm_call", "model": "claude-sonnet-4-6", "provider": "anthropic"},
    "llm:config_change": {"event_type": "config_change", "field": "max_tokens"},
    "llm:circuit_change": {"event_type": "circuit_change", "state": "OPEN"},
}

for ev, data in SAMPLE_PAYLOADS.items():
    await probe_bus.emit(ev, data, agent_did="did:arc:demo:operator/abcd")

print("events captured:", len(probe_log))
print("first 5 captured events:")
for name, data in probe_log[:5]:
    print(f"  {name:>34}  {data}")


The catch list confirms that **the bus is content-agnostic** — it dispatches whatever name + dict you emit. The "taxonomy" is a contract enforced by the *callers* in `arcagent.core.*`, not by the bus itself. This matters for two reasons:

1. **Modules can subscribe to events that haven't fired yet.** No registration error if the name is mistyped — silence. Always assert at least one event lands during integration tests.
2. **Custom modules can introduce their own namespaces** (e.g. `myorg:metrics_flush`) and the bus will dispatch them. Use a colon-prefixed namespace — never collide with `agent:*`, `capability:*`, or `llm:*`.

In [ ]:
# Distribution by namespace
buckets = Counter(name.split(":", 1)[0] for name, _ in probe_log)
for ns, count in sorted(buckets.items()):
    print(f"  {ns:>12} {count:>3} events")


## 3. Module lifecycle — startup, dispatch, shutdown

A module's life on the bus has four observable phases:

```
register_module(m)         ──▶  m sits in the registration list (no startup yet)
                                 │
await bus.startup(ctx)     ──▶  for m in modules:  await m.startup(ctx)   # registration order
                                 │  └─ inside startup: bus.subscribe(...) wires handlers
                                 ▼
await bus.emit(event, data) ──▶  group handlers by priority, fire each group
                                 │  ├─ priority 10 group   → asyncio.gather(...)
                                 │  ├─ priority 50 group   → asyncio.gather(...)
                                 │  └─ priority 100 group  → asyncio.gather(...)
                                 │  (timeouts and exceptions are isolated per handler)
                                 ▼
await bus.shutdown()        ──▶  for m in reversed(modules):  await m.shutdown()
```

Three guarantees worth memorising:

1. **Startup runs in registration order.** If module B's startup needs to subscribe to an event that module A's startup might emit, register A first.
2. **Shutdown runs in reverse registration order.** Last in, first out — the same teardown pattern as Python's `contextlib.ExitStack`.
3. **Handlers within the same priority level run concurrently** (`asyncio.gather`), but priorities are processed sequentially low-to-first. A handler at priority `10` always finishes before any handler at priority `50` starts, even if it's the slowest one in its group.

Here's a concrete demonstration. Three modules — `Alpha`, `Beta`, `Gamma` — each with simple startup/shutdown logging. We register them in order, run lifecycle, observe the trace.

In [ ]:
class TraceModule:
    """Module that records its own lifecycle events into a shared log."""

    def __init__(self, name: str, log: list[str]) -> None:
        self._name = name
        self._log = log

    @property
    def name(self) -> str:
        return self._name

    async def startup(self, ctx: ModuleContext) -> None:
        self._log.append(f"startup:{self._name}")

    async def shutdown(self) -> None:
        self._log.append(f"shutdown:{self._name}")


lifecycle_log: list[str] = []
lc_bus = ModuleBus()
for n in ("alpha", "beta", "gamma"):
    lc_bus.register_module(TraceModule(n, lifecycle_log))

stub_ctx = ModuleContext(
    bus=lc_bus,
    tool_registry=MagicMock(),
    config=MagicMock(),
    telemetry=MagicMock(),
    workspace=MagicMock(),
    llm_config=MagicMock(),
)

await lc_bus.startup(stub_ctx)
await lc_bus.shutdown()

print("lifecycle trace:")
for line in lifecycle_log:
    print(f"  {line}")


Two things are worth highlighting in that trace:

- Startups run **alpha → beta → gamma** (registration order).
- Shutdowns run **gamma → beta → alpha** (reverse).

This isn't decoration — it's required for clean teardown. If `gamma` registered listeners during its `startup` that depend on `alpha`'s telemetry sink still being open, then `alpha`'s `shutdown` must run *after* `gamma`'s — which is exactly what reversed iteration gives you.

**Handler dispatch ordering** is a separate axis from module lifecycle. Inside a single emit, handlers are sorted by `priority` and dispatched in groups. Demonstration:

In [ ]:
dispatch_order: list[str] = []


async def make_h(label: str) -> Any:
    async def handler(ctx: EventContext) -> None:
        dispatch_order.append(label)
    return handler


prio_bus = ModuleBus()
prio_bus.subscribe("ev", await make_h("p100-default"), priority=100)
prio_bus.subscribe("ev", await make_h("p050-security"), priority=50)
prio_bus.subscribe("ev", await make_h("p010-policy"), priority=10)
prio_bus.subscribe("ev", await make_h("p200-logging"), priority=200)
prio_bus.subscribe("ev", await make_h("p010-policy-2"), priority=10)

await prio_bus.emit("ev", {})

print("dispatch order:")
for label in dispatch_order:
    print(f"  {label}")


Notice the two `p010-policy*` handlers can appear in either order relative to each other (same priority → concurrent), but they always finish before any `p050` handler starts.

**Convention** (used by Arc's built-in modules):

| Priority | Use |
|---|---|
| `10` | Policy / authorization (must run first; can veto on `agent:pre_tool`) |
| `50` | Security / sanitisation (input redaction, output filtering) |
| `100` | **Default** — observation, telemetry, capability bridging |
| `200` | Logging, audit, "slow" sinks (don't block faster handlers) |

## 4. HookEntry vs BackgroundTaskEntry vs LifecycleEntry

These three look superficially similar — they all live in `arcagent.core.capability_registry` and they're all "things a module can register" — but they answer **different questions**:

| Type | Question it answers | Lifecycle | Cardinality |
|---|---|---|---|
| **`HookEntry`** | "When **event X** fires, run **this function**." | One-shot — invoked per event emission. | Many per event (fan-out, ordered by `priority`). |
| **`BackgroundTaskEntry`** | "Run this coroutine **continuously** for the agent's lifetime." | Long-running — `asyncio.Task` started on registration; cancelled-and-awaited on replace or shutdown. | One per name. |
| **`LifecycleEntry`** | "When the **agent** starts/stops, call **this object's** `setup`/`teardown`." | Two-shot — `setup(ctx)` once, `teardown()` once. | One per name. |

In other words:

- **Hooks** are *event-driven*. They sleep until an event fires.
- **Background tasks** are *time-driven*. They spin in their own task, doing whatever work needs to happen on a schedule, with their own backpressure.
- **Lifecycle entries** are *state-bound*. They model objects that need explicit acquire/release semantics — file handles, connection pools, telemetry exporters.

In `03-policy-and-modules.ipynb` we wrote a class implementing the `Module` Protocol directly. That's still the most common path for in-process modules. The capability registry types are the *declarative* layer that the **capability loader** uses when scanning skill/extension folders — `@hook(event="X")`, `@background_task(...)`, `@capability(...)` decorators produce these entries, and the loader bridges them onto the bus on the agent's behalf.

Let's look at the data shape of each. Field names tell you what each entry is responsible for tracking — start by inspecting the dataclass fields.

In [ ]:
def show_fields(cls: type) -> None:
    print(f"{cls.__name__}:")
    for f in fields(cls):
        print(f"    {f.name:>14}: {f.type}")

show_fields(HookEntry)
print()
show_fields(BackgroundTaskEntry)
print()
show_fields(LifecycleEntry)


Decision matrix — when to reach for each:

```
          ┌────────────────────────────────────────────────┐
          │ Is the work triggered by something the agent   │
          │ already does (a tool call, a settings change,  │
          │ a turn boundary)?                              │
          └─────────────────┬──────────────────────────────┘
                            │ yes
                            ▼
                    ┌──────────────┐
                    │  HookEntry   │  @hook(event="agent:post_tool")
                    └──────────────┘
                            │ no
                            ▼
          ┌────────────────────────────────────────────────┐
          │ Is the work continuous/periodic with its own   │
          │ schedule, independent of agent activity?       │
          └─────────────────┬──────────────────────────────┘
                            │ yes
                            ▼
                    ┌─────────────────────────┐
                    │  BackgroundTaskEntry    │  @background_task(interval=30)
                    └─────────────────────────┘
                            │ no
                            ▼
          ┌────────────────────────────────────────────────┐
          │ Does the work need explicit setup/teardown to  │
          │ acquire and release a resource?                │
          └─────────────────┬──────────────────────────────┘
                            │ yes
                            ▼
                    ┌─────────────────────┐
                    │  LifecycleEntry     │  @capability  (class with setup/teardown)
                    └─────────────────────┘
```

A *module* (the `Module` Protocol class with `name`, `startup`, `shutdown`) is essentially a single `LifecycleEntry` that internally registers any number of `HookEntry` / `BackgroundTaskEntry` instances during its `startup`. The capability surface lets you skip the wrapper class and declare those primitives directly.

## 5. Pattern: a metrics module

**Goal.** Operator wants per-tool counts and timings, plus error totals. The numbers must be cheap to read at any moment (for a `/metrics` endpoint, a status page, or a Prometheus scrape).

**Design.** Subscribe to the three events that carry the relevant signal:

- `agent:pre_tool` → increment a "tools called" counter, stamp a start time on the trace_id.
- `agent:post_tool` → take elapsed (already in payload as `duration`), add to the histogram, increment "tools succeeded".
- `agent:error` → increment "errors total", bucket by `error_type`.

Expose a single `snapshot()` method that returns a frozen dict — operators read it however they like.

In [ ]:
class MetricsModule:
    """Per-tool counts + duration sums + error totals.

    Reads:  agent:pre_tool, agent:post_tool, agent:error
    Writes: in-memory counters; expose via snapshot().
    """

    def __init__(self) -> None:
        self.calls: Counter[str] = Counter()
        self.successes: Counter[str] = Counter()
        self.duration_sum: dict[str, float] = defaultdict(float)
        self.errors: Counter[str] = Counter()

    @property
    def name(self) -> str:
        return "metrics"

    async def startup(self, ctx: ModuleContext) -> None:
        ctx.bus.subscribe(
            "agent:pre_tool", self._on_pre_tool, module_name=self.name
        )
        ctx.bus.subscribe(
            "agent:post_tool", self._on_post_tool, module_name=self.name
        )
        ctx.bus.subscribe(
            "agent:error", self._on_error, module_name=self.name
        )

    async def shutdown(self) -> None:
        return None

    async def _on_pre_tool(self, ctx: EventContext) -> None:
        self.calls[ctx.data.get("tool", "?")] += 1

    async def _on_post_tool(self, ctx: EventContext) -> None:
        tool = ctx.data.get("tool", "?")
        self.successes[tool] += 1
        self.duration_sum[tool] += float(ctx.data.get("duration", 0.0))

    async def _on_error(self, ctx: EventContext) -> None:
        self.errors[ctx.data.get("error_type", "Unknown")] += 1

    def snapshot(self) -> dict[str, Any]:
        return {
            "calls": dict(self.calls),
            "successes": dict(self.successes),
            "duration_sum_seconds": dict(self.duration_sum),
            "errors": dict(self.errors),
        }


Wire it onto a bus and drive a synthetic workload — three calls to `read_file`, two to `bash`, one transient error.

In [ ]:
metrics_bus = ModuleBus()
metrics = MetricsModule()
metrics_bus.register_module(metrics)

stub_ctx = ModuleContext(
    bus=metrics_bus,
    tool_registry=MagicMock(),
    config=MagicMock(),
    telemetry=MagicMock(),
    workspace=MagicMock(),
    llm_config=MagicMock(),
)
await metrics_bus.startup(stub_ctx)


async def fake_dispatch(bus: ModuleBus, tool: str, duration: float) -> None:
    await bus.emit("agent:pre_tool", {"tool": tool, "args": {}})
    await bus.emit(
        "agent:post_tool", {"tool": tool, "result": "ok", "duration": duration}
    )


for d in (0.012, 0.020, 0.011):
    await fake_dispatch(metrics_bus, "read_file", d)
for d in (0.250, 0.310):
    await fake_dispatch(metrics_bus, "bash", d)

# Synthetic error
await metrics_bus.emit(
    "agent:error",
    {"task": "...", "error": "kaboom", "error_type": "TimeoutError"},
)

snap = metrics.snapshot()
print("calls          :", snap["calls"])
print("successes      :", snap["successes"])
print("duration_sum_s :", {k: round(v, 4) for k, v in snap["duration_sum_seconds"].items()})
print("errors         :", snap["errors"])


**Operational note.** Metrics modules should be **priority 200** so they don't sit in front of policy or security handlers — observation belongs at the back of the priority queue. Wire `priority=200` if you ship this for real.

## 6. Pattern: a structured logging module

**Goal.** Every interesting event turns into a structured log line going to a sink. The sink could be a file, a `JsonlSink`, a `SignedChainSink` from `arctrust.audit`, or stdout in dev. The module shouldn't care — it should fan out to whatever the operator hands it.

**Cross-reference.** This is the *same shape* as `arctrust.audit.emit(AuditEvent, sink)` in `arctrust/04-audit-sinks.ipynb`. The bus-level logger is for *operational* visibility (what the agent did and when); the audit sink is for *forensic* visibility (tamper-evident, schema-frozen). Use both. They answer different questions.

In [ ]:
@dataclass(frozen=True)
class LogRecord:
    timestamp: float
    event: str
    agent_did: str
    payload: dict[str, Any]


class InMemorySink:
    """Stand-in for JsonlSink / SignedChainSink. Tests use this; prod swaps for real."""

    def __init__(self) -> None:
        self.records: list[LogRecord] = []

    def write(self, record: LogRecord) -> None:
        self.records.append(record)


class StructuredLoggerModule:
    """Fan every interesting bus event out to a sink as a structured record."""

    # Events the operator actually cares about — keep this list small.
    SUBSCRIBED_EVENTS = (
        "agent:pre_tool",
        "agent:post_tool",
        "agent:error",
        "agent:settings_changed",
        "capability:added",
        "capability:replaced",
        "capability:removed",
    )

    def __init__(self, sink: InMemorySink) -> None:
        self._sink = sink

    @property
    def name(self) -> str:
        return "structured_logger"

    async def startup(self, ctx: ModuleContext) -> None:
        for ev in self.SUBSCRIBED_EVENTS:
            ctx.bus.subscribe(
                ev, self._log, priority=200, module_name=self.name
            )

    async def shutdown(self) -> None:
        return None

    async def _log(self, ctx: EventContext) -> None:
        self._sink.write(
            LogRecord(
                timestamp=time.monotonic(),
                event=ctx.event,
                agent_did=ctx.agent_did,
                payload=dict(ctx.data),
            )
        )


Stand it up, drive a representative workload, and inspect the sink.

In [ ]:
log_bus = ModuleBus()
sink = InMemorySink()
logger_mod = StructuredLoggerModule(sink)
log_bus.register_module(logger_mod)

await log_bus.startup(
    ModuleContext(
        bus=log_bus,
        tool_registry=MagicMock(),
        config=MagicMock(),
        telemetry=MagicMock(),
        workspace=MagicMock(),
        llm_config=MagicMock(),
    )
)

# Drive synthetic events the operator would care about
DID = "did:arc:demo:operator/abcd1234"
await log_bus.emit("agent:pre_tool", {"tool": "read_file", "args": {"path": "x"}}, agent_did=DID)
await log_bus.emit("agent:post_tool", {"tool": "read_file", "result": "...", "duration": 0.014}, agent_did=DID)
await log_bus.emit(
    "agent:settings_changed",
    {"key": "context.max_tokens", "old_value": 200_000, "new_value": 400_000},
    agent_did=DID,
)
await log_bus.emit(
    "capability:added",
    {"kind": "tool", "name": "search", "version": "1.0.0"},
    agent_did=DID,
)
await log_bus.emit(
    "agent:error",
    {"task": "...", "error": "kaboom", "error_type": "TimeoutError"},
    agent_did=DID,
)

print(f"sink records: {len(sink.records)}")
for r in sink.records:
    print(f"  {r.event:>26}  did={r.agent_did[-8:]}  payload_keys={sorted(r.payload)}")


**Why a separate module instead of "just call telemetry directly"?** Two reasons:

1. **Separation of concerns.** `AgentTelemetry` is a *core* component that emits a fixed set of audit events. The structured logger is *operator-facing*: which events to capture, the destination sink, the redaction policy — all of that is configurable per deployment without touching core.
2. **Hot-swappable sinks.** Federal swaps `InMemorySink` for `arctrust.audit.SignedChainSink`. Personal swaps it for stdout. Same module body, different config. The bus is the integration seam.

## 7. Pattern: a streaming module (events to an external transport)

**Goal.** Forward live events to an external transport — a websocket, a NATS subject, a queue worker. Anything that needs to see what the agent is doing **in real time**, not after the run finishes.

**Design constraints.**

- The bus dispatches handlers under a 30-second default timeout. A blocking transport call would burn that budget. **Push to a local queue, drain in a background task.**
- The transport may be down. Drop or buffer? Operator policy. We model both.
- Backpressure must not stall the bus. If the queue is full, drop and increment a counter — never await a long send while a tool dispatch is waiting on the post_tool emit.

In [ ]:
class StreamingModule:
    """Forward events to an external transport via a bounded asyncio.Queue.

    Hot path (in-handler) is non-blocking: enqueue or drop.
    Drain runs as a background task, started in startup().
    """

    SUBSCRIBED_EVENTS = ("agent:pre_tool", "agent:post_tool", "agent:error")

    def __init__(self, transport: list[dict[str, Any]], *, capacity: int = 16) -> None:
        # Mock transport: a list. In production this is a websocket, NATS, etc.
        self._transport = transport
        self._queue: asyncio.Queue[dict[str, Any]] = asyncio.Queue(maxsize=capacity)
        self._dropped = 0
        self._drain_task: asyncio.Task[None] | None = None

    @property
    def name(self) -> str:
        return "streaming"

    @property
    def dropped(self) -> int:
        return self._dropped

    async def startup(self, ctx: ModuleContext) -> None:
        for ev in self.SUBSCRIBED_EVENTS:
            ctx.bus.subscribe(ev, self._enqueue, module_name=self.name)
        self._drain_task = asyncio.create_task(self._drain_loop())

    async def shutdown(self) -> None:
        if self._drain_task is not None:
            self._drain_task.cancel()
            try:
                await self._drain_task
            except asyncio.CancelledError:
                pass

    async def _enqueue(self, ctx: EventContext) -> None:
        envelope = {"event": ctx.event, "data": dict(ctx.data), "agent_did": ctx.agent_did}
        try:
            self._queue.put_nowait(envelope)
        except asyncio.QueueFull:
            self._dropped += 1

    async def _drain_loop(self) -> None:
        while True:
            envelope = await self._queue.get()
            # In production this would be `await ws.send_json(envelope)` etc.
            # The drain task can take as long as it needs — it's not on the
            # hot path, so a slow transport doesn't slow tool dispatch.
            self._transport.append(envelope)
            self._queue.task_done()


Drive a workload — emit faster than the drain task can keep up to demonstrate backpressure handling.

In [ ]:
stream_bus = ModuleBus()
transport: list[dict[str, Any]] = []
streaming = StreamingModule(transport, capacity=4)
stream_bus.register_module(streaming)

await stream_bus.startup(
    ModuleContext(
        bus=stream_bus,
        tool_registry=MagicMock(),
        config=MagicMock(),
        telemetry=MagicMock(),
        workspace=MagicMock(),
        llm_config=MagicMock(),
    )
)

# Burst more events than capacity to exercise the drop path
for i in range(20):
    await stream_bus.emit(
        "agent:pre_tool", {"tool": "echo", "args": {"i": i}}, agent_did="did:demo"
    )

# Let the drain task catch up
await asyncio.sleep(0.05)

print("transport received :", len(transport))
print("queue dropped      :", streaming.dropped)
print("first 3 forwarded  :")
for env in transport[:3]:
    print(f"  {env['event']}  {env['data']}")

await stream_bus.shutdown()


**Production checklist** for a streaming module like this:

- Use a **bounded** queue. Unbounded queues mask leaks and blow up memory.
- Decide drop policy explicitly. Drop newest? Drop oldest? Coalesce? Operator-tunable via config.
- Emit a `metrics:queue_depth` event (your own namespace!) at intervals so the metrics module can see backpressure.
- On `agent:shutdown`, *drain* the queue before cancelling the task — otherwise tail events are lost.

## 8. Failure modes

Three failure surfaces deserve explicit demonstration. The bus is designed so that **a buggy module never crashes the agent**, but you must understand exactly what "contained" means in each case.

### 8.1 Module raises during startup

The bus catches the exception, **logs it**, emits a `module_lifecycle` event with outcome `"error"` to the UI reporter, and **moves on to the next module**. The failed module's handlers are never registered, so any events the failed module *would have* observed silently flow past.

This means: **always** verify your module's startup completed cleanly — either by checking `bus.handler_count(event)` for the events you expected to subscribe to, or by emitting a sentinel event from the module's startup body and asserting it landed.

In [ ]:
class BrokenStartup:
    """Module that raises during startup — simulates a vault/identity failure."""

    @property
    def name(self) -> str:
        return "broken"

    async def startup(self, ctx: ModuleContext) -> None:
        raise RuntimeError("vault unreachable; cannot acquire signing key")

    async def shutdown(self) -> None:
        return None


class HealthyModule:
    @property
    def name(self) -> str:
        return "healthy"

    async def startup(self, ctx: ModuleContext) -> None:
        ctx.bus.subscribe("ev", self._h, module_name=self.name)

    async def shutdown(self) -> None:
        return None

    async def _h(self, ctx: EventContext) -> None:
        return None


fail_bus = ModuleBus()
fail_bus.register_module(BrokenStartup())
fail_bus.register_module(HealthyModule())

# bus.startup must NOT raise — it logs and continues.
try:
    await fail_bus.startup(
        ModuleContext(
            bus=fail_bus,
            tool_registry=MagicMock(),
            config=MagicMock(),
            telemetry=MagicMock(),
            workspace=MagicMock(),
            llm_config=MagicMock(),
        )
    )
    print("module failed during startup: BrokenStartup raised RuntimeError, bus.startup() completed")
except Exception as exc:
    print("UNEXPECTED — bus.startup raised:", exc)

# Verify HealthyModule still registered.
print("'ev' handlers     :", fail_bus.handler_count("ev"))
print("modules in bus    :", [m.name for m in fail_bus._modules])


**Note** the broken module is still in `bus._modules` — the registry tracks it for `shutdown()` (some modules need teardown even if startup partially failed). What's missing is its **handlers**: nothing got subscribed because `startup` didn't reach the `subscribe` call.

If you want to **fail loud** when a critical module's startup fails — for example when you need the vault to be alive before the agent can do anything useful — wrap startup in your *own* try/except at the agent level and raise `ModuleBusError` explicitly:

In [ ]:
# Pattern: explicit fail-loud on critical-module startup.
# We catch ModuleBusError below so the notebook still runs end-to-end —
# in your own code you'd let it propagate to the agent's caller.
critical_bus = ModuleBus()
critical_bus.register_module(BrokenStartup())

try:
    # Manually invoke each critical module's startup outside bus.startup
    # so the exception propagates instead of being swallowed.
    critical = critical_bus.get_module("broken")
    if critical is not None:
        try:
            await critical.startup(
                ModuleContext(
                    bus=critical_bus,
                    tool_registry=MagicMock(),
                    config=MagicMock(),
                    telemetry=MagicMock(),
                    workspace=MagicMock(),
                    llm_config=MagicMock(),
                )
            )
        except RuntimeError as exc:
            raise ModuleBusError(
                code="CRITICAL_MODULE_STARTUP_FAILED",
                message=f"Module 'broken' failed; refusing to continue: {exc}",
                details={"module": "broken"},
            ) from exc
except ModuleBusError as exc:
    print("fail-loud surfaced:", exc)
    print("code             :", exc.code)
    print("details          :", exc.details)


### 8.2 Handler raises during emit

Same containment story, different scope. The bus catches the exception inside `_run_handler`, logs it, and continues. Other handlers in the same priority group still complete, the next priority group still runs, the emit caller still gets a non-vetoed `EventContext` back.

**The implication:** handler exceptions are **invisible** to upstream code. If your metrics module crashes on `agent:post_tool`, you won't see broken metrics in the agent's call site — you'll see *missing* numbers. Always log exceptions at the module level too if you care about observability.

In [ ]:
raise_log: list[str] = []


async def good_handler(ctx: EventContext) -> None:
    raise_log.append("good ran")


async def bad_handler(ctx: EventContext) -> None:
    raise_log.append("bad ran (will raise next)")
    raise RuntimeError("simulated handler crash")


async def also_good_handler(ctx: EventContext) -> None:
    raise_log.append("also_good ran")


emit_bus = ModuleBus()
emit_bus.subscribe("ev", good_handler)
emit_bus.subscribe("ev", bad_handler)
emit_bus.subscribe("ev", also_good_handler)

result = await emit_bus.emit("ev", {})
print("module failed during emit: bad_handler raised, others kept running")
print("captured trace :", raise_log)
print("emit returned  :", type(result).__name__, "vetoed=", result.is_vetoed)


### 8.3 Handler times out

Handlers run under a per-registration timeout (`timeout_seconds`, default 30s). A timeout is logged at `WARNING` and the handler is dropped — same as a crash. The next group still runs.

This is **why metrics/logging modules belong at priority 200**: a slow logging sink should never starve a faster security check at priority 50.

In [ ]:
timeout_log: list[str] = []


async def slow(ctx: EventContext) -> None:
    timeout_log.append("slow started")
    await asyncio.sleep(1.0)  # > 100ms timeout below
    timeout_log.append("slow finished — should NOT appear")


async def fast(ctx: EventContext) -> None:
    timeout_log.append("fast ran")


timeout_bus = ModuleBus()
timeout_bus.subscribe("ev", slow, timeout_seconds=0.05)
timeout_bus.subscribe("ev", fast, priority=200)

t0 = time.monotonic()
await timeout_bus.emit("ev", {})
elapsed_ms = (time.monotonic() - t0) * 1000
print(f"emit elapsed: {elapsed_ms:.0f} ms (slow handler hit 50ms timeout)")
print("trace:", timeout_log)


### 8.4 Mapping exception types to surfaces

Quick reference card:

| Trigger | Surface | Subclass of | Bus behaviour |
|---|---|---|---|
| Policy DENY | `PolicyDenied` | `ArcAgentError` | Bus never sees the call. |
| `ctx.veto(reason)` on `agent:pre_tool` | `ToolVetoedError` | `ToolError` → `ArcAgentError` | Bus dispatched all handlers; first veto wins. |
| Bus lifecycle assertion (custom) | `ModuleBusError` | `ArcAgentError` | Caller-emitted; bus does not raise this on its own. |
| Handler crashes / times out | (logged, swallowed) | n/a | All other handlers in the same priority group still run. |
| Module startup raises | (logged, swallowed) | n/a | Module is kept in registry for shutdown; handlers not subscribed. |

`ModuleBusError` is **not** raised by the bus on handler failure — that's the whole point of the isolation. It exists so *callers* (the agent, your operational tooling) can wrap bus interactions in structured exceptions when they need fail-loud semantics.

In [ ]:
err = ModuleBusError(
    code="CRITICAL_MODULE_STARTUP_FAILED",
    message="Module 'vault' failed to start: vault unreachable",
    details={"module": "vault", "phase": "startup"},
)
print("str(err)  :", err)
print("code      :", err.code)
print("component :", err.component)
print("details   :", err.details)


## 9. Composition — multiple modules in concert

Real deployments stack three or more modules. The interesting questions become:

- **What's the dispatch order?** (priorities, then registration order within a priority)
- **What does each module see?** (they all see the same `EventContext` snapshot — but mutations to `ctx.data` in one handler do not leak to the next emit)
- **How do you debug "wrong number" or "missing event"?** (`bus.handler_count(event)` and per-module module_name tagging)

Let's run the metrics module, the logger, and the streaming module together against a synthetic workload that mimics one tool-heavy agent run.

In [ ]:
# Compose all three production modules.
combo_bus = ModuleBus()
combo_metrics = MetricsModule()
combo_sink = InMemorySink()
combo_logger = StructuredLoggerModule(combo_sink)
combo_transport: list[dict[str, Any]] = []
combo_streaming = StreamingModule(combo_transport, capacity=32)

# Order matters — metrics first (cheapest), logger next, streaming last.
combo_bus.register_module(combo_metrics)
combo_bus.register_module(combo_logger)
combo_bus.register_module(combo_streaming)

await combo_bus.startup(
    ModuleContext(
        bus=combo_bus,
        tool_registry=MagicMock(),
        config=MagicMock(),
        telemetry=MagicMock(),
        workspace=MagicMock(),
        llm_config=MagicMock(),
    )
)

print("modules         :", [m.name for m in combo_bus._modules])
for ev in ("agent:pre_tool", "agent:post_tool", "agent:error", "agent:settings_changed"):
    print(f"  {ev:>24} handlers: {combo_bus.handler_count(ev)}")


Drive a small synthetic run — five tool calls, one error, one settings change. Each event fans out to all three modules in parallel (within priority), then we read each module's view.

In [ ]:
DID = "did:arc:demo:operator/abcd1234"

for i in range(3):
    await combo_bus.emit("agent:pre_tool", {"tool": "read_file", "args": {"i": i}}, agent_did=DID)
    await combo_bus.emit("agent:post_tool", {"tool": "read_file", "result": "ok", "duration": 0.01}, agent_did=DID)

for i in range(2):
    await combo_bus.emit("agent:pre_tool", {"tool": "bash", "args": {"i": i}}, agent_did=DID)
    await combo_bus.emit("agent:post_tool", {"tool": "bash", "result": "ok", "duration": 0.25}, agent_did=DID)

await combo_bus.emit("agent:error", {"task": "x", "error": "boom", "error_type": "RuntimeError"}, agent_did=DID)
await combo_bus.emit(
    "agent:settings_changed",
    {"key": "context.max_tokens", "old_value": 200_000, "new_value": 400_000},
    agent_did=DID,
)

# Let the streaming drain task catch up
await asyncio.sleep(0.05)

print("=== metrics ===")
print(combo_metrics.snapshot())
print()
print("=== structured log records ===")
print(f"count: {len(combo_sink.records)}")
for r in combo_sink.records[-3:]:
    print(f"  ... {r.event:>26}  payload_keys={sorted(r.payload)}")
print()
print("=== streaming transport ===")
print(f"forwarded: {len(combo_transport)}, dropped: {combo_streaming.dropped}")

await combo_bus.shutdown()


**Debugging composition.** When something looks off, walk this checklist:

1. `bus.handler_count(event)` — is the count what you expect for that event?
2. Are handlers tagged with `module_name=...` so the bus's debug log identifies *which* module's handler crashed/timed-out?
3. Is the failing module's `startup` actually completing? (See §8.1 — silent startup failures = silent missing handlers.)
4. Are priorities collision-free? Two priority-100 handlers run concurrently; if they race on shared state, you have a bug *in the modules*, not the bus.
5. Is your handler async? Sync handlers can't be awaited and the bus's `_run_handler` will raise `TypeError` on first dispatch (logged, swallowed).

## 10. Summary

**Event taxonomy.** `ArcAgent` emits a fixed catalog of events through the `ModuleBus`: `agent:init`, `agent:ready`, `agent:pre_respond` / `post_respond`, `agent:error`, `agent:pre_tool` / `post_tool`, `agent:pre_plan` / `post_plan`, `agent:assemble_prompt`, `agent:pre_compaction`, `agent:settings_changed`, `agent:tools_reloaded`, `agent:shutdown`, `capability:added` / `replaced` / `removed` / `registration_failed` / `registration_warning` / `setup_failed`, `llm:call_complete` / `config_change` / `circuit_change`. **Only `agent:pre_tool` accepts a veto.** Everything else is observe-only.

**Module lifecycle.** `register_module → startup (registration order) → emit (priority groups, concurrent within a group) → shutdown (reverse order)`. Within an emit, priority `10` runs before `50` runs before `100` runs before `200`; same priority handlers run concurrently via `asyncio.gather`.

**Three "things you can register".**

| Concept | Trigger | Cardinality |
|---|---|---|
| `HookEntry` | Bus event fires | Many per event |
| `BackgroundTaskEntry` | Continuously, on its own schedule | One per name |
| `LifecycleEntry` | Agent startup / shutdown | One per name |

A `Module` Protocol class is the explicit form of `LifecycleEntry`-with-internal-hooks. The capability registry types are the *declarative* form the loader produces from `@hook` / `@background_task` / `@capability` decorators.

**Three production patterns covered.**

- **Metrics** — counters and durations, exposed via `snapshot()`. Priority 200.
- **Structured logger** — fans events out to a pluggable sink (`InMemorySink` here, `JsonlSink` / `SignedChainSink` from `arctrust.audit` in production).
- **Streaming** — bounded `asyncio.Queue` + background drain task, never blocks the hot path.

**Failure containment.** Handler crashes, handler timeouts, and module-startup failures are all caught, logged, and isolated — they never raise out of `bus.emit` or `bus.startup`. `ModuleBusError` exists for *callers* to raise when they want fail-loud semantics on top of bus operations; the bus itself does not throw it.

**API recap** — the surface this notebook exercised:

- **`arcagent.core.module_bus`** — `ModuleBus`, `Module` (Protocol), `EventContext`, `ModuleContext`, `subscribe`, `emit`, `register_module`, `get_module`, `startup`, `shutdown`, `handler_count`, `handler_count_by_module`, `unsubscribe_by_module_prefix`.
- **`arcagent.core.capability_registry`** — `HookEntry`, `BackgroundTaskEntry`, `LifecycleEntry`.
- **`arcagent`** — `ToolVetoedError`, `ModuleBusError`.

**Where to next.**

- For the operational story of how arctrust audit sinks slot into a logging module, see [`arctrust/04-audit-sinks.ipynb`](../arctrust/04-audit-sinks.ipynb).
- For the policy pipeline that runs *before* the bus on every tool call, see [`arctrust/03-policy-pipeline.ipynb`](../arctrust/03-policy-pipeline.ipynb).
- For the run loop that produces most of these events (`arcrun.run`, the `tool.start` / `tool.end` / `turn.start` / `turn.end` bridge), see [`arcrun/01-core-react.ipynb`](../arcrun/01-core-react.ipynb).